# Run 3 (1 year): a 1-day gap

1. Gap: run 2's series set to zero from day 12 to 13; the likelihood leaves out the time pixels it
   touches plus one on each side.
2. Simulate (same seed as run 2, so the same series outside the gap).
3. Wavelet sampler, everything else as run 2.
4. Compare with run 2: is $P_c(t)$ still recovered, and how much wider are the bands?

Needs runs 1 and 2. Change the gap with `GAP` (and `--gap` on the command line, [README](README.md)).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import corner

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'simulations' / 'common'))
import injection as I
import diagnostics as D
from bahamas.wavelet.model import DAY
from bahamas.wavelet.simulate import power_modulation

RERUN = False
CFG = I.load_config(ROOT / 'simulations' / '1_year' / 'config' / 'wavelet.yaml')
GAP = (12.0, 13.0)                      # days
CFG_GAP = I.with_gap(GAP, buffer_pixels=1, config=CFG)
OUT = Path(I.out_dir(CFG)) / 'run3'
CH = ['A', 'E']
amp_true, amp_ref = I.true_amp(), I.load_reference(I.reference_path(CFG))['amp']   # reference fit: run 1

## 1–2. Gap and data

In [ ]:
data2, inj = I.load_or_simulate('sky', 'run2', CFG)
like2, post2 = I.build_posterior(data2, CFG)
data, _ = I.load_or_simulate('sky', 'run3', CFG_GAP, rerun=RERUN)
like, post = I.build_posterior(data, CFG_GAP)
dT = data.Nf * data.dt
off = np.flatnonzero(~I.gap_time_mask(data, CFG_GAP))
gap_lo, gap_hi = (data.times[off[0]] - dT / 2) / DAY, (data.times[off[-1]] + dT / 2) / DAY
print(f'left out: {len(off)} time pixels, {gap_lo:.2f}-{gap_hi:.2f} d; pixels used: run 2 {like2.npixels}, '
      f'run3 {like.npixels} ({100 * (1 - like.npixels / like2.npixels):.1f}% fewer)')

x, dt, t0 = I.load_timeseries('run3', CFG_GAP)
tt = (t0 + np.arange(x.shape[1]) * dt) / DAY
zoom = (tt > GAP[0] - 5) & (tt < GAP[1] + 5)
fig, ax = plt.subplots(figsize=(11, 2.6))
ax.plot(tt[zoom][::20], x[0, zoom][::20], lw=0.3)
ax.axvspan(gap_lo, gap_hi, color='0.5', alpha=0.3, label='left out of the likelihood')
ax.set_xlabel('time [days]'); ax.set_ylabel('A'); ax.legend()

## 3. Wavelet sampler

In [ ]:
res_path = OUT / 'chains.npz'
if RERUN or not res_path.exists():
    I.run('sky', 'run3', CFG_GAP)
r = dict(np.load(res_path, allow_pickle=True))
r2 = dict(np.load(Path(I.out_dir(CFG)) / 'run2' / 'chains.npz', allow_pickle=True))
print(f"runtime {float(r['runtime_s']) / 60:.0f} min, acceptance {np.mean(r['acceptance_fraction']):.2f}")
D.report(r, post, np.array([np.mean(GAP), 60., 180., 300.]) * DAY);

## 4. Gap vs no gap

In [ ]:
td = r['modulation_times'] / DAY
near, far = np.abs(td - np.mean(GAP)) < (GAP[1] - GAP[0]) / 2 + 1, np.abs(td - np.mean(GAP)) > (GAP[1] - GAP[0]) / 2 + 5
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for i, c in enumerate(CH):
    for name, res in [('no gap (run 2)', r2), ('gap (run3)', r)]:
        rel = res['modulation_draws'][:, i] / res['P_true'][i]          # recovered / injected
        q = np.percentile(rel, [5, 50, 95], axis=0)
        axes[i].fill_between(td, q[0] - 1, q[2] - 1, alpha=0.35, label=f'{name} 90%')
        w = q[2] - q[0]
        print(f'{c} {name:15s} 90% width: {100 * np.median(w[far]):.1f}% away from the gap, '
              f'up to {100 * w[near].max():.1f}% at it')
    axes[i].axvspan(gap_lo, gap_hi, color='0.5', alpha=0.3)
    axes[i].axhline(0, color='k', ls='--', lw=1); axes[i].set_ylabel(f'{c}: recovered / injected − 1')
axes[0].legend(fontsize=8); axes[1].set_xlabel('time [days]')
fig.savefig(OUT / 'modulation_gap_vs_nogap.png', dpi=150)

In [ ]:
names = [str(n).split(':')[-1] for n in r['psd_names']]
fig = corner.corner(r2['psd_chain'], labels=names, color='C0', plot_datapoints=False, levels=(0.5, 0.9))
corner.corner(r['psd_chain'], fig=fig, color='C1', truths=r['psd_injected'], plot_datapoints=False, levels=(0.5, 0.9))
fig.legend(handles=[plt.Line2D([], [], color='C0', label='no gap (run 2)'), plt.Line2D([], [], color='C1', label='gap')])
fig.savefig(OUT / 'corner_gap_vs_nogap.png', dpi=110);